In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)

target = "converted"
features = df.drop(columns=[target]).copy()

categorical = features.select_dtypes(include=["object", "str"]).columns.tolist()
numerical = features.select_dtypes(exclude=["object", "str"]).columns.tolist()

df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)

print("Q1 Mode of industry:", df["industry"].mode()[0])

corr_matrix = df[numerical].corr().abs()
candidate_pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]

for col1, col2 in candidate_pairs:
    print(f"{col1} and {col2}: {corr_matrix.loc[col1, col2]:.4f}")

best_pair = max(candidate_pairs, key=lambda pair: corr_matrix.loc[pair[0], pair[1]])
print("Q2 Highest correlation pair:", best_pair)

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

del df_train[target]
del df_val[target]
del df_test[target]

all_features = categorical + numerical

mi_scores = {}
for col in categorical:
    score = mutual_info_score(y_train, df_train[col])
    mi_scores[col] = round(score, 2)
    print(f"MI score for {col}: {round(score, 2)}")

print("Q3 Highest MI score:", max(mi_scores, key=mi_scores.get))

dv = DictVectorizer(sparse=False)
train_dict = df_train[all_features].to_dict(orient="records")
X_train = dv.fit_transform(train_dict)

val_dict = df_val[all_features].to_dict(orient="records")
X_val = dv.transform(val_dict)

model = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
original_accuracy = accuracy_score(y_val, y_pred)
print("Q4 Validation accuracy:", round(original_accuracy, 2))

elimination_candidates = [
    "lead_source",
    "number_of_courses_viewed",
    "interaction_count",
]

differences = {}
for col in elimination_candidates:
    subset = [feature for feature in all_features if feature != col]

    dv_sub = DictVectorizer(sparse=False)
    X_train_sub = dv_sub.fit_transform(df_train[subset].to_dict(orient="records"))
    X_val_sub = dv_sub.transform(df_val[subset].to_dict(orient="records"))

    model_sub = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)
    model_sub.fit(X_train_sub, y_train)

    acc_sub = accuracy_score(y_val, model_sub.predict(X_val_sub))
    diff = original_accuracy - acc_sub
    differences[col] = diff
    print(f"Without {col} -> Accuracy: {acc_sub:.4f}, Difference: {diff:.4f}")

print("Q5 Smallest difference feature:", min(differences, key=lambda k: abs(differences[k])))

c_values = [0.000001, 0.00001, 0.0001, 0.001]
c_scores = {}

for c in c_values:
    model_c = LogisticRegression(solver="liblinear", C=c, max_iter=1000, random_state=42)
    model_c.fit(X_train, y_train)

    acc_c = round(accuracy_score(y_val, model_c.predict(X_val)), 3)
    c_scores[c] = acc_c
    print(f"C={c} -> Accuracy: {acc_c}")

best_c = max(c_values, key=lambda c: (c_scores[c], -c))
print("Q6 Best C parameter:", best_c)

Q1 Mode of industry: technology
interaction_count and lead_score: 0.9157
number_of_courses_viewed and lead_score: 0.7572
number_of_courses_viewed and interaction_count: 0.7216
annual_income and interaction_count: 0.1228
Q2 Highest correlation pair: ('interaction_count', 'lead_score')
MI score for lead_source: 0.03
MI score for industry: 0.0
MI score for employment_status: 0.02
MI score for location: 0.0
Q3 Highest MI score: lead_source
Q4 Validation accuracy: 0.65
Without lead_source -> Accuracy: 0.6420, Difference: 0.0030
Without number_of_courses_viewed -> Accuracy: 0.6430, Difference: 0.0020
Without interaction_count -> Accuracy: 0.6010, Difference: 0.0440
Q5 Smallest difference feature: number_of_courses_viewed
C=1e-06 -> Accuracy: 0.598
C=1e-05 -> Accuracy: 0.598
C=0.0001 -> Accuracy: 0.613
C=0.001 -> Accuracy: 0.645
Q6 Best C parameter: 0.001
